## Json Parsing for one file

In [0]:
df=(spark.read
  .option("multiline", "true")
  .json("s3://openaq-project-dm/raw/locations/page_1.json")
)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, explode_outer

df_flat = (
    df
    .withColumn("result", explode_outer(col("results")))
    .withColumn("instrument", explode_outer(col("result.instruments")))
    .withColumn("sensor", explode_outer(col("result.sensors")))
    .withColumn("license", explode_outer(col("result.licenses")))
    .select(
        # Location
        col("result.id").alias("location_id"),
        col("result.name").alias("location_name"),
        col("result.locality"),
        col("result.timezone"),
        col("result.distance"),

        # Coordinates
        col("result.coordinates.latitude").alias("latitude"),
        col("result.coordinates.longitude").alias("longitude"),

        # Country
        col("result.country.id").alias("country_id"),
        col("result.country.code").alias("country_code"),
        col("result.country.name").alias("country_name"),

        # Owner / Provider
        col("result.owner.id").alias("owner_id"),
        col("result.owner.name").alias("owner_name"),
        col("result.provider.id").alias("provider_id"),
        col("result.provider.name").alias("provider_name"),

        # DateTimes
        col("result.datetimeFirst.utc").alias("first_seen_utc"),
        col("result.datetimeLast.utc").alias("last_seen_utc"),

        # Flags
        col("result.isMobile"),
        col("result.isMonitor"),

        # Instrument (exploded)
        col("instrument.id").alias("instrument_id"),
        col("instrument.name").alias("instrument_name"),

        # Sensor (exploded)
        col("sensor.id").alias("sensor_id"),
        col("sensor.name").alias("sensor_name"),
        col("sensor.parameter.id").alias("parameter_id"),
        col("sensor.parameter.name").alias("parameter_name"),
        col("sensor.parameter.units").alias("parameter_units"),
        col("sensor.parameter.displayName").alias("parameter_displayName"),

        # License (exploded)
        col("license.id").alias("license_id"),
        col("license.name").alias("license_name"),
        col("license.dateFrom"),
        col("license.dateTo"),
        col("license.attribution.name").alias("license_attribution_name"),
        col("license.attribution.url").alias("license_attribution_url")
    )
)

In [0]:
display(df_flat)

#  For multiple files that gets added in the S3's folder

In [0]:
dbutils.widgets.text("Entity","demo","Please Enter Entity name")

In [0]:
table_name = dbutils.widgets.get("Entity")

In [0]:
from pyspark.sql.functions import col, explode_outer
from pyspark.sql.types import *

##  Creating Schema

In [0]:
schema = StructType([
    StructField("results", ArrayType(
        StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
            StructField("locality", StringType()),
            StructField("timezone", StringType()),
            StructField("distance", StringType()),

            StructField("coordinates", StructType([
                StructField("latitude", DoubleType()),
                StructField("longitude", DoubleType())
            ])),

            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("code", StringType()),
                StructField("name", StringType())
            ])),

            StructField("owner", StructType([
                StructField("id", LongType()),
                StructField("name", StringType())
            ])),

            StructField("provider", StructType([
                StructField("id", LongType()),
                StructField("name", StringType())
            ])),

            StructField("instruments", ArrayType(
                StructType([
                    StructField("id", LongType()),
                    StructField("name", StringType())
                ])
            )),

            StructField("sensors", ArrayType(
                StructType([
                    StructField("id", LongType()),
                    StructField("name", StringType()),
                    StructField("parameter", StructType([
                        StructField("id", LongType()),
                        StructField("name", StringType()),
                        StructField("units", StringType()),
                        StructField("displayName", StringType())
                    ]))
                ])
            )),

            StructField("licenses", ArrayType(
                StructType([
                    StructField("id", LongType()),
                    StructField("name", StringType()),
                    StructField("dateFrom", StringType()),
                    StructField("dateTo", StringType()),
                    StructField("attribution", StructType([
                        StructField("name", StringType()),
                        StructField("url", StringType())
                    ]))
                ])
            )),

            StructField("datetimeFirst", StructType([
                StructField("utc", StringType()),
                StructField("local", StringType())
            ])),

            StructField("datetimeLast", StructType([
                StructField("utc", StringType()),
                StructField("local", StringType())
            ])),

            StructField("isMobile", BooleanType()),
            StructField("isMonitor", BooleanType())
        ])
    ))
])

## Using Autoloader

In [0]:
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema)
    .load(f"s3://openaq-project-dm/raw/{table_name}/")
)

In [0]:
df_flat = (
    df_stream
    .withColumn("result", explode_outer(col("results")))
    .withColumn("instrument", explode_outer(col("result.instruments")))
    .withColumn("sensor", explode_outer(col("result.sensors")))
    .withColumn("license", explode_outer(col("result.licenses")))
    .select(
        # Location
        col("result.id").alias("location_id"),
        col("result.name").alias("location_name"),
        col("result.locality"),
        col("result.timezone"),
        col("result.distance"),

        # Coordinates
        col("result.coordinates.latitude").alias("latitude"),
        col("result.coordinates.longitude").alias("longitude"),

        # Country
        col("result.country.id").alias("country_id"),
        col("result.country.code").alias("country_code"),
        col("result.country.name").alias("country_name"),

        # Owner / Provider
        col("result.owner.id").alias("owner_id"),
        col("result.owner.name").alias("owner_name"),
        col("result.provider.id").alias("provider_id"),
        col("result.provider.name").alias("provider_name"),

        # Dates
        col("result.datetimeFirst.utc").alias("first_seen_utc"),
        col("result.datetimeLast.utc").alias("last_seen_utc"),

        # Flags
        col("result.isMobile"),
        col("result.isMonitor"),

        # Instruments
        col("instrument.id").alias("instrument_id"),
        col("instrument.name").alias("instrument_name"),

        # Sensors
        col("sensor.id").alias("sensor_id"),
        col("sensor.name").alias("sensor_name"),
        col("sensor.parameter.id").alias("parameter_id"),
        col("sensor.parameter.name").alias("parameter_name"),
        col("sensor.parameter.units").alias("parameter_units"),
        col("sensor.parameter.displayName").alias("parameter_displayName"),

        # Licenses
        col("license.id").alias("license_id"),
        col("license.name").alias("license_name"),
        col("license.dateFrom"),
        col("license.dateTo"),
        col("license.attribution.name").alias("license_attribution_name"),
        col("license.attribution.url").alias("license_attribution_url")
    )
)


In [0]:
(
    df_flat.writeStream
    .format("delta")
    .option("checkpointLocation", "s3://openaq-project-dm/checkpoints/locations/")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"openaq.bronze.brz_{table_name}")
)

In [0]:
display(df_flat, checkpointLocation="s3://openaq-project-dm/checkpoints/debug/")

### Licenses

In [0]:
df=(spark.read
  .option("multiline", "true")
  .json("s3://openaq-project-dm/raw/Licenses/page_1.json")
)

In [0]:
df.printSchema()

In [0]:
schema_Licenses=StructType([
    StructField("results",ArrayType(
        StructType([
            StructField("attributionRequired",BooleanType()),
            StructField("commercialUseAllowed",BooleanType()),
            StructField("id",LongType()),
            StructField("modificationAllowed",BooleanType()),
            StructField("name",StringType()),
            StructField("redistributionAllowed",BooleanType()),
            StructField("shareAlikeRequired",BooleanType()),
            StructField("sourceUrl",StringType())        
    ]))
)])

In [0]:
df_Licenses_stream=(
    spark.readStream.format("cloudFiles").option("cloudFiles.format", "json")
    .schema(schema_Licenses)
    .load(f"s3://openaq-project-dm/raw/{table_name}/")
)

In [0]:
df_Licenses_stream=(df_Licenses_stream
                    .withColumn("result",explode_outer(col("results")))
                    .select(
                        col("result.id").alias("license_id"),
                        col("result.attributionRequired").alias("attributionRequired"),
                        col("result.commercialUseAllowed").alias("commercialUseAllowed"),
                        col("result.modificationAllowed").alias("modificationAllowed"),
                        col("result.name").alias("license_name"),
                        col("result.redistributionAllowed").alias("redistributionAllowed"),
                        col("result.shareAlikeRequired").alias("shareAlikeRequired"),
                        col("result.sourceUrl").alias("sourceUrl")

))

In [0]:
(
    df_Licenses_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "s3://openaq-project-dm/Licenses/checkpoints/")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"openaq.bronze.brz_{table_name}")
)

### Instruments

In [0]:
df=(spark.read
  .option("multiline", "true")
  .json("s3://openaq-project-dm/raw/Instruments/page_1.json")
)

In [0]:
df.printSchema()

In [0]:
schema_Instruments = StructType([
    StructField("results",ArrayType(
        StructType([
            StructField("id",LongType()),
            StructField("isMonitor",BooleanType()),
            # Manufacturer StructField
            StructField("manufacturer",StructType([
                StructField("id",LongType()), 
                StructField("name",StringType())
        ]))
])))])

In [0]:
df_instruments_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema_Instruments)
    .load(f"s3://openaq-project-dm/raw/{table_name}/")
)

In [0]:
df_instruments_stream = (
    df_instruments_stream
    .withColumn("result", explode_outer(col("results")))
    .select(
        col("result.id").alias("instrument_id"),
        col("result.isMonitor").alias("isMonitor"),
        col("result.manufacturer.id").alias("manufacturer_id"),
        col("result.manufacturer.name").alias("manufacturer_name")
))

In [0]:
(
    df_instruments_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "s3://openaq-project-dm/raw/Instruments/checkpoints/")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"openaq.bronze.brz_{table_name}")
)